<a href="https://colab.research.google.com/github/smuhrj/Understanding-Uncertainty-HJones/blob/main/Lab07_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inference Lab: Pricing Under Uncertainty


**Run this notebook in Google Colab.** The `demand` module is compiled for Linux and Python 3.13 only, so it will not import on a Mac or Windows machine.

The goal of this lab is to use what you've learned in this class to find the optimal price for something. You can't know for sure what the underlying curve of revenue vs price looks like. All you know is what you would know in the real world, information you gain by sampling-- you can market test any price using the demand function below. When you set the price, the market test runs for a certain amount of time, and then reports responses-- a list of 1s (bought) and 0s (didn't buy) for that price point. The total number in the sample, $n$, is therefore a random variable, given by `len(responses)`. The number of consumers who buy the product is `responses.sum().` Every time you run a market test, it takes time (literally, it will take a while to run, so don't do this lab last minute!).

In [2]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand

responses = demand(price=14.0)

print(len(responses), responses.sum())

/content/demand_lab
167 128


## 1. Compare two prices

Run one market test with price = 30. Then test price = 25. Which price earns more revenue per potential customer?

1. For each price, estimate the proportion $\hat{p}$ of customers who buy, and the revenue per customer $\hat{r}$.
2. Estimate the standard error of each $\hat{r}$ by bootstrapping.
3. Compute a 95% confidence interval for each $\hat{r}$.
4. Compute a 95% confidence interval for the difference $\hat{r}_{25} - \hat{r}_{30}$.
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.



In [3]:
responses30 = demand(price=30.0)
responses25 = demand(price=25.0)

print("Price 30:", len(responses30), responses30.sum())
print("Price 25:", len(responses25), responses25.sum())

Price 30: 160 65
Price 25: 190 67


In [4]:
p30 = responses30.mean()
p25 = responses25.mean()

r30 = 30 * p30
r25 = 25 * p25

print("p30 =", p30)
print("r30 =", r30)

print("p25 =", p25)
print("r25 =", r25)

p30 = 0.40625
r30 = 12.1875
p25 = 0.3526315789473684
r25 = 8.81578947368421


In [6]:
import numpy as np

R = 10000
rng = np.random.default_rng(123)

boot_r30 = []
boot_r25 = []

for i in range(R):
    sample30 = rng.choice(responses30, size=len(responses30), replace=True)
    sample25 = rng.choice(responses25, size=len(responses25), replace=True)

    boot_r30.append(30 * sample30.mean())
    boot_r25.append(25 * sample25.mean())

boot_r30 = np.array(boot_r30)
boot_r25 = np.array(boot_r25)

In [7]:
se30 = boot_r30.std(ddof=1)
se25 = boot_r25.std(ddof=1)

se30, se25

(np.float64(1.1597948276518457), np.float64(0.8680546866288928))

In [8]:
ci30 = (r30 - 1.96*se30, r30 + 1.96*se30)
ci25 = (r25 - 1.96*se25, r25 + 1.96*se25)

ci30, ci25

((np.float64(9.914302137802382), np.float64(14.460697862197618)),
 (np.float64(7.114402287891581), np.float64(10.51717665947684)))

In [9]:
diff = r25 - r30
diff

np.float64(-3.3717105263157894)

In [10]:
boot_diff = boot_r25 - boot_r30
se_diff = boot_diff.std(ddof=1)
ci_diff = (diff - 1.96*se_diff, diff + 1.96*se_diff)

se_diff, ci_diff

(np.float64(1.4512177561682205),
 (np.float64(-6.216097328405501), np.float64(-0.5273237242260773)))

The proportion at 30 is 0.406 and the propotion at 25 is 0.353. This means per customer, you are likely to have an increase in profit of 12.19 at 30 and 8.82 at 25. Therefor there is likely more profit at a price of 30. The bootstrap SEs are 1.16 for 30 and 0.87 for 25. The confidence intervals are: CI30: [9.91, 14.46] and CI25: [7.11, 10.52]. The estimated difference between 25 and 30 is about -3.37 with a confidence interval of [-6.22, -0.53].

I am 95% confident that the profit at the price of 30 is in the range [9.91, 14.46]

I am 95% confident that the profit at the price of 25 is in the range [7.11, 10.52]

I am 95% confident that the difference in profit per customer between 25 and 30 is [-6.22, -0.53] with 30 being more profitable.

## 2. The cost of sampling

7. Run a second market test at each price. Pool the two price = 30 samples (`np.concatenate`), and pool the two price = 25 samples.
8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.
   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?


In [11]:
responses30_2 = demand(price=30.0)
responses25_2 = demand(price=25.0)

print("Second Price 30:", len(responses30_2), responses30_2.sum())
print("Second Price 25:", len(responses25_2), responses25_2.sum())

Second Price 30: 160 72
Second Price 25: 131 48


In [12]:
responses30_pool = np.concatenate([responses30, responses30_2])
responses25_pool = np.concatenate([responses25, responses25_2])

print(len(responses30_pool), responses30_pool.sum())
print(len(responses25_pool), responses25_pool.sum())

320 137
321 115


In [13]:
p30_pool = responses30_pool.mean()
p25_pool = responses25_pool.mean()

r30_pool = 30 * p30_pool
r25_pool = 25 * p25_pool

r30_pool, r25_pool

(np.float64(12.84375), np.float64(8.95638629283489))

In [14]:
R = 10000
rng = np.random.default_rng(123)

boot_r30_pool = []
boot_r25_pool = []

for i in range(R):
    sample30 = rng.choice(
        responses30_pool,
        size=len(responses30_pool),
        replace=True)

    sample25 = rng.choice(
        responses25_pool,
        size=len(responses25_pool),
        replace=True)

    boot_r30_pool.append(30 * sample30.mean())
    boot_r25_pool.append(25 * sample25.mean())
boot_r30_pool = np.array(boot_r30_pool)
boot_r25_pool = np.array(boot_r25_pool)

In [15]:
se30_pool = boot_r30_pool.std(ddof=1)
se25_pool = boot_r25_pool.std(ddof=1)

se30_pool, se25_pool

(np.float64(0.8378541254535252), np.float64(0.6676637476516781))

In [16]:
ci30_pool = (r30_pool - 1.96*se30_pool,r30_pool + 1.96*se30_pool)
ci25_pool = (r25_pool - 1.96*se25_pool, r25_pool + 1.96*se25_pool)
ci30_pool, ci25_pool

((np.float64(11.20155591411109), np.float64(14.48594408588891)),
 (np.float64(7.647765347437602), np.float64(10.26500723823218)))

In [17]:
diff_pool = r25_pool - r30_pool

boot_diff_pool = boot_r25_pool - boot_r30_pool
se_diff_pool = boot_diff_pool.std(ddof=1)

ci_diff_pool = (diff_pool - 1.96*se_diff_pool, diff_pool + 1.96*se_diff_pool)

diff_pool, se_diff_pool, ci_diff_pool

(np.float64(-3.8873637071651093),
 np.float64(1.0741220218645406),
 (np.float64(-5.992642870019608), np.float64(-1.7820845443106097)))

In [18]:
se30 / se30_pool, se25 / se25_pool

(np.float64(1.3842443361176457), np.float64(1.30013751635016))

The pooled estimates are 12.84 for 30 and 8.96 for 25. The bootstrap SE decreased to 0.84 for 30 and 0.67 for 25. The 95% confidence intervals were [11.20, 14.49] for 30 and [7.65, 10.27] for 25. The difference is -3.89, with a 95% confidence interval of [-5.99, -1.78] so $30 is still more profitable.

The SEs shrank by about 1.30 and 1.38 for 25 and 30, which makes sense because the sample sizes were doubled and the formuala has the square root of n in the denominator. the square root of 2 is about 1.4 which is why the SEs shrank by about that factor.

Increasing the sample increased my certianty of my previous result and did not change the conclusion. While it is helpful to know this, it was a result we could have predicted if we had more data, so it was not neccissary. The second test costed time and was the same as the first, so we did not need it.

## 3. Finding the optimal price

Now find the price that maximizes revenue per customer. Search prices between about \$5 and \$100.

Every market test takes time, and time spent testing is revenue lost: while you test, you aren't selling at your best price. Your confidence intervals tell you whether another test is worth running. **Use as few market tests as you can (<25 would be good).** Your grade will be based on, in this order, whether you:
- accurately used the information you have by accurately interpreting confidence intervals
- found a good price, close to the optimal price
- and used fewer tests.

If this sounds fun to you, run with it! If you are still very confused about what to do, read the hints below.

<details><Summary>Hints</summary>
<b>Hint 1:</b> There are a few ways you could search:

- Brute force: test every price on a fine grid. This takes a market test at every price, including prices that are obviously bad. This would be a bad choice.
- Hill climbing: compare the CI for two prices, move toward the better one, and repeat. For example, start from the 25/30 comparison above; based on which price is better, choose another price \$10 away in that direction (15 if 25 was better; 40 if 30 was better). Shrink the step size as you get close to a good price. If you can't detect a difference between two prices, either shrink the step size or increase the sample size (i.e. run another market test at the same price).
- other strategies that I won't mention here.
  
<b>Hint 2:</b> You may want to try starting your tests with a few different prices across the range; there may be local maxima you will get stuck in if you just hill climb towards a single maximum.

<b>Hint 3:</b> You can make up your own strategy or try a mix of strategies. However, keep in mind that your grade will reflect how well you ran A/B testing and interpreting confidence intervals, more than how complex your strategy is. Similarly, a lucky guess with a brute force strategy would not be a good reproducible strategy.

<b>Hint 4:</b>
Turn your code from part 1 into a function, so you can easily input responses, run bootstrapping, calculate SE, and calculate confidence intervals. Then make another function where you give it two confidence intervals and make a decision based on them: sample more at the same test, or move in one direction with some step size.
   </details>

**Before you run any tests,** write down your plan: which prices you'll test first, how you'll decide where to go next, and what your intervals need to show before you stop. Then follow it, and note anywhere you changed course and why.

For any method, stop when your intervals show that another test is unlikely to change your answer by enough to matter.

**Deliverable:** A ~2-3 paragraph write up of the strategy you chose, your final recommended price, a 95% CI for $r$ at that price, the number of market tests you ran, and one or two sentences on how your intervals told you when to stop.

My plan is to start above 30 since we know a price of 30 is already more profitable than 25. I will then run the test in intervals and once the profit decreases, I will search in the maxamized interval for a specific answer. The intervals I will do will be spaced out in tens and I will make a looping function to also save time rather than copying and pasting the same codes in a repetative way.

In [19]:
def test_price(price, R=10000):
    responses = demand(price=float(price))

    p_hat = responses.mean()
    r_hat = price * p_hat

    boot = []

    for i in range(R):
        sample = np.random.choice(
            responses,
            size=len(responses),
            replace=True)
        boot.append(price * sample.mean())

    boot = np.array(boot)

    se = boot.std(ddof=1)

    ci = (
        r_hat - 1.96 * se,
        r_hat + 1.96 * se)

    return {
        "price": price,
        "n": len(responses),
        "buyers": responses.sum(),
        "p_hat": p_hat,
        "r_hat": r_hat,
        "se": se,
        "ci_low": ci[0],
        "ci_high": ci[1],
        "responses": responses
    }

In [20]:
result40 = test_price(40)
result40

{'price': 40,
 'n': 166,
 'buyers': np.int64(62),
 'p_hat': np.float64(0.37349397590361444),
 'r_hat': np.float64(14.939759036144578),
 'se': np.float64(1.5129860795719823),
 'ci_low': np.float64(11.974306320183493),
 'ci_high': np.float64(17.905211752105664),
 'responses': array([1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1,
        0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0,
        1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
        0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
        1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0,
        0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0], dtype=int8)}

In [21]:
result50 = test_price(50)
result50

{'price': 50,
 'n': 180,
 'buyers': np.int64(52),
 'p_hat': np.float64(0.28888888888888886),
 'r_hat': np.float64(14.444444444444443),
 'se': np.float64(1.6876733217038022),
 'ci_low': np.float64(11.13660473390499),
 'ci_high': np.float64(17.752284154983894),
 'responses': array([0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
        1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
        1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0,
        1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0,
        0, 0, 0, 0], dtype=int8)}

In [22]:
result60 = test_price(60)
result60

{'price': 60,
 'n': 186,
 'buyers': np.int64(48),
 'p_hat': np.float64(0.25806451612903225),
 'r_hat': np.float64(15.483870967741936),
 'se': np.float64(1.92970779343912),
 'ci_low': np.float64(11.70164369260126),
 'ci_high': np.float64(19.266098242882613),
 'responses': array([1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1,
        0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8)}

In [23]:
result70 = test_price(70)
result70

{'price': 70,
 'n': 154,
 'buyers': np.int64(37),
 'p_hat': np.float64(0.24025974025974026),
 'r_hat': np.float64(16.818181818181817),
 'se': np.float64(2.4055864937375997),
 'ci_low': np.float64(12.103232290456122),
 'ci_high': np.float64(21.533131345907513),
 'responses': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       dtype=int8)}

In [24]:
result80 = test_price(80)
result80

{'price': 80,
 'n': 203,
 'buyers': np.int64(46),
 'p_hat': np.float64(0.22660098522167488),
 'r_hat': np.float64(18.12807881773399),
 'se': np.float64(2.3515071268287184),
 'ci_low': np.float64(13.519124849149701),
 'ci_high': np.float64(22.737032786318277),
 'responses': array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1,
        0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0,
        0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0], dtype=int8)}

In [25]:
result90 = test_price(90)
result90

{'price': 90,
 'n': 176,
 'buyers': np.int64(38),
 'p_hat': np.float64(0.2159090909090909),
 'r_hat': np.float64(19.431818181818183),
 'se': np.float64(2.8040682067071803),
 'ci_low': np.float64(13.935844496672111),
 'ci_high': np.float64(24.927791866964256),
 'responses': array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
        0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0],
       dtype=int8)}

In [26]:
result100 = test_price(100)
result100

{'price': 100,
 'n': 240,
 'buyers': np.int64(12),
 'p_hat': np.float64(0.05),
 'r_hat': np.float64(5.0),
 'se': np.float64(1.4064439104983508),
 'ci_low': np.float64(2.2433699354232326),
 'ci_high': np.float64(7.756630064576767),
 'responses': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0,

In [27]:
result95 = test_price(95)
result95

{'price': 95,
 'n': 156,
 'buyers': np.int64(17),
 'p_hat': np.float64(0.10897435897435898),
 'r_hat': np.float64(10.352564102564102),
 'se': np.float64(2.360624597615585),
 'ci_low': np.float64(5.725739891237556),
 'ci_high': np.float64(14.979388313890649),
 'responses': array([1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0], dtype=int8)}

In [28]:
result85 = test_price(85)
result85

{'price': 85,
 'n': 185,
 'buyers': np.int64(32),
 'p_hat': np.float64(0.17297297297297298),
 'r_hat': np.float64(14.702702702702704),
 'se': np.float64(2.380193631960988),
 'ci_low': np.float64(10.037523184059166),
 'ci_high': np.float64(19.367882221346242),
 'responses': array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
        0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 0], dtype=int8)}

In [29]:
result92 = test_price(92)
result92

{'price': 92,
 'n': 180,
 'buyers': np.int64(28),
 'p_hat': np.float64(0.15555555555555556),
 'r_hat': np.float64(14.311111111111112),
 'se': np.float64(2.476464092590396),
 'ci_low': np.float64(9.457241489633937),
 'ci_high': np.float64(19.164980732588287),
 'responses': array([0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
        0, 0, 0, 1], dtype=int8)}

In [30]:
result88 = test_price(88)
result88

{'price': 88,
 'n': 194,
 'buyers': np.int64(32),
 'p_hat': np.float64(0.16494845360824742),
 'r_hat': np.float64(14.515463917525773),
 'se': np.float64(2.357145503630051),
 'ci_low': np.float64(9.895458730410873),
 'ci_high': np.float64(19.135469104640674),
 'responses': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0,
        0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1,
        1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0], dtype=int8)}

I ran the tests increasing in 10 dollar incraments for the price. This way, I avoided testing every number between 5 and 100. I also started at 30 since I knew 30 already had a higher profit than 25. Once I hit 100, the profit dropped which meant the optimal price must be between 80 and 100. I then tested 95 since it was in the middle of 90-100 and found that the optimal price must be less so I then tried 85. From here on I dialed in on numbers between 85 and 95 to find that the optimal price is 90 dollars. I did this instead of creating a full loop since that would process through all price options between 5 and 100 and that would have been too much memory to use.

Looking back at it now, I likely did not need all of the additional information in each loop since the projected profit is the only thing that mattered in this simulation. All in all, I only used 11 tests to find that the optimal price is 90 and this still used less tests than testing each number. The 95% confidence interval for profit per customer at the price of 90 is [13.94, 24.93]. There were no more needed tests because I tested values around the price at 90 and none came close to the projected profit per customer at 19.43 dollars.

Note: Used AI for formatting and confirming I had the write code to make the output clean. Also needed help making the loop and output for Q3 so that I could automattically punch in the number and it woult output the profit along with the other neccessary results.